# 1 · Convert the raw Excel exports

Reads `data/raw/series_<n>.xlsx` and writes one CSV per sausage to `data/converted/`
(`series_<n>_sausage_<k>.csv`). Each file holds the uplinks of that sausage's two probes
(LSE01, LTC-FS) plus the shared WS202 uplinks, in arrival order: the stream the models see.

* Series 1 is annotated (column `label`: Umroetung, Reifung Phase 1, Reifung Phase 2). Series 2 is not.
* Air temperature: the ambient device (LSN50-v2-S31) is not added as rows. Every uplink gets the most recent air
  temperature in `ambient_temperature`. **Check the reported column and value range below**: the two exports name it
  differently, and for series 2 the column is an assumption (`AMBIENT_COLUMNS` in `ml_core/conversion.py`).
* Dropped: pH probes and air humidity.
* The logic lives in `ml_core/conversion.py` (tested in `tests/test_ml_core.py`).

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))  # project root, so ml_core is importable
import matplotlib.pyplot as plt
import pandas as pd
from ml_core import data
from ml_core.conversion import convert_all

In [ ]:
RAW_DIR = data.RAW_DIR
CONVERTED_DIR = data.CONVERTED_DIR

In [ ]:
summary = convert_all(RAW_DIR, CONVERTED_DIR)
summary

## Sanity check: one sausage from each run

Series 2 starts right after series 1, so both fit on one time axis. Top: moisture, coloured by phase (series 1) or grey
(series 2, no labels). Middle: conductivity. Bottom: sausage core temperature (LTC-FS) and air temperature.

In [ ]:
SAMPLES = ["series_1_sausage_1", "series_2_sausage_1"]

fig, axes = plt.subplots(3, 1, figsize=(13, 8), sharex=True)
for name in SAMPLES:
    df = pd.read_csv(os.path.join(CONVERTED_DIR, f"{name}.csv"), parse_dates=["timestamp"])
    if "ambient_temperature" not in df.columns:
        raise RuntimeError("This CSV was written by an older conversion without air temperature. "
                           "Restart the kernel (ml_core may still be the old version) and run all cells again.")
    lse, probe = df[df["sensor"] == "lse01"], df[df["sensor"] == "ltcfs"]
    labels = lse["label"].fillna("") if "label" in lse else pd.Series("", index=lse.index)
    for label, part in lse.groupby(labels, sort=False):
        axes[0].scatter(part["timestamp"], part["lse01_moisture"], s=1, label=f"{name} · {label or 'no labels'}",
                        color=None if label else "grey")
    axes[1].scatter(lse["timestamp"], lse["lse01_conductivity"], s=1, label=name)
    axes[2].plot(probe["timestamp"], probe["ltcfs_temperature"], lw=.5, label=f"{name} · sausage (LTC-FS)")
    axes[2].plot(probe["timestamp"], probe["ambient_temperature"], lw=.5, color="black", alpha=.5,
                 label="air" if name == SAMPLES[0] else None)
axes[0].set(ylabel="LSE01 moisture (%)")
axes[1].set(ylabel="LSE01 conductivity (µS/cm)")
axes[2].set(ylabel="°C")
for ax in axes:
    ax.legend(markerscale=8, fontsize=8, loc="upper left")
plt.show()